# ETL — Variables Macroeconómicas

**Fase:** Ingeniería del Dato 

## 1. Importaciones y rutas
Importo las librerías que necesitaré en el cuaderno y ruteo a los archivos

In [1]:
import pandas as pd
import numpy as np
import os
import sqlite3
from sqlalchemy import create_engine, Date
import warnings
warnings.filterwarnings('ignore')

BASE      = os.path.expanduser('~/Library/Mobile Documents/com~apple~CloudDocs/UFV/UNIVERSIDAD FRANCISCO DE VITORIA/4º/TFG')
MACRO_DIR = os.path.join(BASE, 'Datos', 'Datos Macro')
DB_PATH   = os.path.join(BASE, 'proyecto', 'data', 'db', 'tfg.db')
PROC_DIR  = os.path.join(BASE, 'proyecto', 'data', 'processed')
engine    = create_engine(f'sqlite:///{DB_PATH}')
print(f'Macro DIR : {MACRO_DIR}')
print(f'DB PATH   : {DB_PATH}')

Macro DIR : /Users/adriancelada/Library/Mobile Documents/com~apple~CloudDocs/UFV/UNIVERSIDAD FRANCISCO DE VITORIA/4º/TFG/Datos/Datos Macro
DB PATH   : /Users/adriancelada/Library/Mobile Documents/com~apple~CloudDocs/UFV/UNIVERSIDAD FRANCISCO DE VITORIA/4º/TFG/proyecto/data/db/tfg.db


## 2. Funciones auxiliares
Parsear a número europeo "." para miles y "," para decimales. Leer datos de Reuters y parsear fechas, Investing.com, INE y pivotar de datos matriz a serie temporal y BCE y pasar a datetime español dd.mm.yyyy

In [8]:
def parsear_numero_europeo(serie):                                                                                                                                                                    
      return (
          serie.astype(str)                                                                                                                                                                             
              .str.replace('%', '', regex=False)
              .str.replace('.', '', regex=False)
              .str.replace(',', '.', regex=False)
              .str.strip()
              .pipe(pd.to_numeric, errors='coerce')
      )

def leer_investing_csv(filepath, col_nombre):
      df = pd.read_csv(filepath)
      df['fecha'] = pd.to_datetime(df['Fecha'], format='%d.%m.%Y', errors='coerce')
      col_src = 'Ultimo' if 'Ultimo' in df.columns else 'Último'
      if col_src not in df.columns:
          col_src = df.columns[1]
      df[col_nombre] = parsear_numero_europeo(df[col_src])
      df = df[['fecha', col_nombre]].dropna().sort_values('fecha').reset_index(drop=True)
      return df[df['fecha'] >= '2005-01-01']

def leer_reuters_macro(filepath, col_nombre, sheet='First Release Data'):
      df_raw = pd.read_excel(filepath, sheet_name=sheet, header=None)
      header_row = None
      for i, row in df_raw.iterrows():
          vals = [str(v) for v in row.values if pd.notna(v)]
          if vals and vals[0].strip() == 'Period':
              header_row = i
              break
      if header_row is None:
          print(f'  AVISO: No se encontro fila Period en {filepath}')
          return None
      df = pd.read_excel(filepath, sheet_name=sheet, header=header_row)
      df.columns = ['periodo', 'release_date', col_nombre] + list(df.columns[3:])
      df = df[['periodo', col_nombre]].copy()
      df = df[df['periodo'] != 'Period']
      df = df.dropna(subset=['periodo', col_nombre])
      df[col_nombre] = pd.to_numeric(df[col_nombre], errors='coerce')
      def parsear_periodo(p):
          p = str(p).strip()
          try:
              if p.startswith('Q'):
                  q, y = p.split(' ')
                  mes = {'Q1':'01','Q2':'04','Q3':'07','Q4':'10'}[q]
                  return pd.to_datetime(f'{y}-{mes}-01')
              else:
                  return pd.to_datetime(p, format='%b %Y')
          except:
              return pd.NaT
      df['fecha'] = df['periodo'].apply(parsear_periodo)
      df = df[['fecha', col_nombre]].dropna().sort_values('fecha').reset_index(drop=True)
      return df[df['fecha'] >= '2005-01-01']

def leer_ine_matriz(filepath, col_nombre):
      df_raw = pd.read_excel(filepath, header=None)
      header_row = None
      for i, row in df_raw.iterrows():
          if 'M01' in [str(v) for v in row.values]:
              header_row = i
              break
      df = pd.read_excel(filepath, header=header_row)
      anyo_col = None
      for col in df.columns:
          nums = pd.to_numeric(df[col], errors='coerce').dropna()
          if len(nums) > 0 and nums.between(1900, 2100).all():
              anyo_col = col
              break
      if anyo_col is None:
          print(f'  AVISO: No se encontró columna de años en {filepath}')
          return pd.DataFrame(columns=['fecha', col_nombre])
      df = df.rename(columns={anyo_col: 'anyo'})
      df['anyo'] = pd.to_numeric(df['anyo'], errors='coerce')
      df = df.dropna(subset=['anyo'])
      df['anyo'] = df['anyo'].astype(int)
      meses = [c for c in df.columns if str(c).startswith('M') and str(c)[1:].isdigit()]
      df_long = df[['anyo'] + meses].melt(id_vars='anyo', var_name='mes', value_name=col_nombre)
      df_long['mes_num'] = df_long['mes'].str[1:].astype(int)
      df_long['fecha'] = pd.to_datetime(
          df_long['anyo'].astype(str) + '-' + df_long['mes_num'].astype(str).str.zfill(2) + '-01'
      )
      df_long[col_nombre] = pd.to_numeric(df_long[col_nombre], errors='coerce')
      df_long = df_long[['fecha', col_nombre]].dropna().sort_values('fecha').reset_index(drop=True)
      return df_long[df_long['fecha'] >= '2005-01-01']

print('Funciones definidas correctamente')

Funciones definidas correctamente


## 3. Bloque 1 — Actividad Económica

Leo los archivos por orden macro, según que reflejen cada uno.

In [9]:
AE_DIR = os.path.join(MACRO_DIR, 'Actividad Económica')                                                                                                                                               
df_pib  = leer_reuters_macro(os.path.join(AE_DIR, 'PIB_ES_YOY.xlsx'), 'pib_yoy')                                                                                                                      
df_pmi  = leer_reuters_macro(os.path.join(AE_DIR, 'PMI SPAIN.xlsx'), 'pmi')                                                                                                                           
df_ipi  = leer_ine_matriz(os.path.join(AE_DIR, 'Spain IPI.xlsx'), 'ipi_yoy')
df_paro = leer_reuters_macro(os.path.join(AE_DIR, 'Tasa de Paro España.xlsx'), 'tasa_paro')                                                                                                           
print(f'PIB       : {len(df_pib)} obs')
print(f'PMI       : {len(df_pmi)} obs')
print(f'IPI       : {len(df_ipi)} obs')
print(f'Tasa Paro : {len(df_paro)} obs')





PIB       : 82 obs
PMI       : 36 obs
IPI       : 249 obs
Tasa Paro : 82 obs


## 4. Bloque 2 — Condiciones Monetarias y Financieras

In [10]:
CM_DIR = os.path.join(MACRO_DIR, 'Condiciones Monetarias y Financieras')
df_bono_es = leer_investing_csv(os.path.join(CM_DIR, 'Bono Español 10 años.csv'), 'bono_es_10y')
df_bono_de = leer_investing_csv(os.path.join(CM_DIR, 'Bono Alemán 10 años.csv'), 'bono_de_10y')
df_spread  = leer_investing_csv(os.path.join(CM_DIR, 'Bono España vs Alemania 10Y.csv'), 'spread_es_de')
print(f'Bono ES   : {len(df_bono_es)} obs')
print(f'Bono DE   : {len(df_bono_de)} obs')
print(f'Spread    : {len(df_spread)} obs')

df_eur3m = pd.read_excel(os.path.join(CM_DIR, 'Euribor 3M.xlsx'), sheet_name='DATA(FM)')
df_eur3m = df_eur3m[['DATE','OBS.VALUE']].rename(columns={'DATE':'fecha','OBS.VALUE':'euribor_3m'})
df_eur3m['fecha'] = pd.to_datetime(df_eur3m['fecha'], errors='coerce')
df_eur3m['euribor_3m'] = pd.to_numeric(df_eur3m['euribor_3m'], errors='coerce')
df_eur3m = df_eur3m.dropna().sort_values('fecha').reset_index(drop=True)
df_eur3m = df_eur3m[df_eur3m['fecha'] >= '2005-01-01']

df_eur6m = pd.read_excel(os.path.join(CM_DIR, 'Euribor 6M.xlsx'), sheet_name='DATA(FM)')
df_eur6m = df_eur6m[['DATE','OBS.VALUE']].rename(columns={'DATE':'fecha','OBS.VALUE':'euribor_6m'})
df_eur6m['fecha'] = pd.to_datetime(df_eur6m['fecha'], errors='coerce')
df_eur6m['euribor_6m'] = pd.to_numeric(df_eur6m['euribor_6m'], errors='coerce')
df_eur6m = df_eur6m.dropna().sort_values('fecha').reset_index(drop=True)
df_eur6m = df_eur6m[df_eur6m['fecha'] >= '2005-01-01']
print(f'Euribor 3M: {len(df_eur3m)} obs')
print(f'Euribor 6M: {len(df_eur6m)} obs')

meses_es = {'Enero':'01','Febrero':'02','Marzo':'03','Abril':'04','Mayo':'05','Junio':'06','Julio':'07','Agosto':'08','Septiembre':'09','Octubre':'10','Noviembre':'11','Diciembre':'12'}
df_bce = pd.read_csv(os.path.join(CM_DIR, 'Tipo Interes BCE.csv'), sep=';', decimal=',')
df_bce.columns = ['anyo','mes','tipo_dfr','tipo_mlf','tipo_mro']
df_bce = df_bce.dropna(subset=['anyo','mes'])
df_bce['mes_num'] = df_bce['mes'].map(meses_es)
df_bce['fecha'] = pd.to_datetime(df_bce['anyo'].astype(int).astype(str)+'-'+df_bce['mes_num']+'-01', errors='coerce')
for col in ['tipo_dfr','tipo_mlf','tipo_mro']:
    df_bce[col] = pd.to_numeric(df_bce[col], errors='coerce')
df_bce = df_bce[['fecha','tipo_dfr','tipo_mlf','tipo_mro']].dropna(subset=['fecha']).sort_values('fecha').reset_index(drop=True)
df_bce = df_bce[df_bce['fecha'] >= '2005-01-01']
print(f'Tipos BCE : {len(df_bce)} obs')

Bono ES   : 4978 obs
Bono DE   : 4777 obs
Spread    : 2321 obs
Euribor 3M: 250 obs
Euribor 6M: 250 obs
Tipos BCE : 46 obs


## 5. Bloque 3 — Precios e Inflación

In [11]:
PI_DIR = os.path.join(MACRO_DIR, 'Precios e Inflación')
df_ipc     = leer_reuters_macro(os.path.join(PI_DIR, 'IPC ESPAÑA.xlsx'), 'ipc_yoy')
df_ipc_sub = leer_ine_matriz(os.path.join(PI_DIR, 'IPC subyacente españa.xlsx'), 'ipc_sub_mom')
print(f'IPC        : {len(df_ipc)} obs')
print(f'IPC Subya. : {len(df_ipc_sub)} obs')

IPC        : 162 obs
IPC Subya. : 250 obs


## 6. Bloque 4 — Riesgo Global, Commodities y Divisas

In [12]:
RG_DIR = os.path.join(MACRO_DIR, 'Riesgo Global Commodities y Divisas')
df_vix    = leer_investing_csv(os.path.join(RG_DIR, 'S&P 500 VIX.csv'), 'vix')
df_vibex  = leer_investing_csv(os.path.join(RG_DIR, 'VIBEX.csv'), 'vibex')
df_vstoxx = leer_investing_csv(os.path.join(RG_DIR, 'VSTOXX EUR.csv'), 'vstoxx')
df_brent  = leer_investing_csv(os.path.join(RG_DIR, 'Barril BRENT.csv'), 'brent')
print(f'VIX    : {len(df_vix)} obs')
print(f'VIBEX  : {len(df_vibex)} obs')
print(f'VSTOXX : {len(df_vstoxx)} obs')
print(f'Brent  : {len(df_brent)} obs')

df_eurusd = pd.read_excel(os.path.join(RG_DIR, 'EUR:USD.xlsx'))
df_eurusd = df_eurusd.rename(columns={df_eurusd.columns[0]:'fecha', df_eurusd.columns[1]:'eur_usd'})
df_eurusd['fecha'] = pd.to_datetime(df_eurusd['fecha'], errors='coerce')
df_eurusd['eur_usd'] = pd.to_numeric(df_eurusd['eur_usd'], errors='coerce')
df_eurusd = df_eurusd.dropna().sort_values('fecha').reset_index(drop=True)
df_eurusd = df_eurusd[df_eurusd['fecha'] >= '2005-01-01']
print(f'EUR/USD: {len(df_eurusd)} obs')

df_raw = pd.read_excel(os.path.join(RG_DIR, 'GAS TTF.xlsx'), header=None)
header_row = None
for i, row in df_raw.iterrows():
    if any('exchange date' in str(v).lower() for v in row.values):
        header_row = i
        break
df_gas = pd.read_excel(os.path.join(RG_DIR, 'GAS TTF.xlsx'), header=header_row)
df_gas = df_gas.rename(columns={'Exchange Date':'fecha', 'Close':'gas_ttf'})
df_gas['fecha'] = pd.to_datetime(df_gas['fecha'], errors='coerce')
df_gas['gas_ttf'] = pd.to_numeric(df_gas['gas_ttf'], errors='coerce')
df_gas = df_gas[['fecha','gas_ttf']].dropna().sort_values('fecha').reset_index(drop=True)
df_gas = df_gas[df_gas['fecha'] >= '2005-01-01']
print(f'Gas TTF: {len(df_gas)} obs')

VIX    : 4989 obs
VIBEX  : 1821 obs
VSTOXX : 3117 obs
Brent  : 4994 obs
EUR/USD: 5441 obs
Gas TTF: 4012 obs


## 7. Consolidación en tablas

Junto los datos en 5 tablas uno por cada bloque de datos macroeconómicos

In [13]:
df_act_mensual = df_ipi.merge(df_pmi, on='fecha', how='outer').merge(df_ipc_sub, on='fecha', how='outer').sort_values('fecha').reset_index(drop=True)
df_act_trimes  = df_pib.merge(df_paro, on='fecha', how='outer').sort_values('fecha').reset_index(drop=True)
df_mon_diario  = df_bono_es.merge(df_bono_de, on='fecha', how='outer').merge(df_spread, on='fecha', how='outer').merge(df_eurusd, on='fecha', how='outer').sort_values('fecha').reset_index(drop=True)
df_mon_mensual = df_eur3m.merge(df_eur6m, on='fecha', how='outer').merge(df_bce, on='fecha', how='outer').merge(df_ipc, on='fecha', how='outer').sort_values('fecha').reset_index(drop=True)
df_riesgo      = df_vix.merge(df_vibex, on='fecha', how='outer').merge(df_vstoxx, on='fecha', how='outer').merge(df_brent, on='fecha', how='outer').merge(df_gas, on='fecha', how='outer').sort_values('fecha').reset_index(drop=True)

print(f'macro_act_mensual : {len(df_act_mensual):>5} filas')
print(f'macro_act_trimes  : {len(df_act_trimes):>5} filas')
print(f'macro_mon_diario  : {len(df_mon_diario):>5} filas')
print(f'macro_mon_mensual : {len(df_mon_mensual):>5} filas')
print(f'macro_riesgo      : {len(df_riesgo):>5} filas')

macro_act_mensual :   250 filas
macro_act_trimes  :    82 filas
macro_mon_diario  :  5865 filas
macro_mon_mensual :   423 filas
macro_riesgo      :  5091 filas


## 8. Control de calidad

Reviso cada tabla a ver el histórico de fechas que contienen y conteo los nulos

In [14]:
tablas = {
    'macro_act_mensual': df_act_mensual,
    'macro_act_trimes' : df_act_trimes,
    'macro_mon_diario' : df_mon_diario,
    'macro_mon_mensual': df_mon_mensual,
    'macro_riesgo'     : df_riesgo
}
for nombre, df in tablas.items():
    nulos = {k:v for k,v in df.isnull().sum().items() if v > 0 and k != 'fecha'}
    print(f'\n{nombre} ({len(df)} filas):')
    print(f'  Nulos: {nulos if nulos else "ninguno"}')
    print(f'  Rango: {df["fecha"].min().date()} -> {df["fecha"].max().date()}')


macro_act_mensual (250 filas):
  Nulos: {'ipi_yoy': 1, 'pmi': 214}
  Rango: 2005-01-01 -> 2025-10-01

macro_act_trimes (82 filas):
  Nulos: ninguno
  Rango: 2005-04-01 -> 2025-07-01

macro_mon_diario (5865 filas):
  Nulos: {'bono_es_10y': 887, 'bono_de_10y': 1088, 'spread_es_de': 3544, 'eur_usd': 424}
  Rango: 2005-01-03 -> 2025-11-11

macro_mon_mensual (423 filas):
  Nulos: {'euribor_3m': 173, 'euribor_6m': 173, 'tipo_dfr': 377, 'tipo_mlf': 377, 'tipo_mro': 377, 'ipc_yoy': 261}
  Rango: 2005-01-31 -> 2025-10-31

macro_riesgo (5091 filas):
  Nulos: {'vix': 102, 'vibex': 3270, 'vstoxx': 1974, 'brent': 97, 'gas_ttf': 1079}
  Rango: 2006-03-01 -> 2025-10-31


## 9. Carga en SQLite y backup CSV

In [15]:
for nombre, df in tablas.items():
    df.to_sql(name=nombre, con=engine, if_exists='replace', index=False, dtype={'fecha': Date()})
    df.to_csv(os.path.join(PROC_DIR, f'{nombre}.csv'), index=False)
    print(f'  OK: {nombre} ({len(df):,} filas)')

  OK: macro_act_mensual (250 filas)
  OK: macro_act_trimes (82 filas)
  OK: macro_mon_diario (5,865 filas)
  OK: macro_mon_mensual (423 filas)
  OK: macro_riesgo (5,091 filas)


## 10. Verificación final

Verifico que se han cargado correctamente las 5 tablas a `tfg.db`

In [16]:
with sqlite3.connect(DB_PATH) as conn:
    ts = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
    print('Todas las tablas en tfg.db:')
    for t in ts['name']:
        n = pd.read_sql(f'SELECT COUNT(*) as n FROM "{t}"', conn).iloc[0,0]
        print(f'  {t:<35} {n:>7} filas')

Todas las tablas en tfg.db:
  macro_act_mensual                       250 filas
  macro_act_trimes                         82 filas
  macro_mon_diario                       5865 filas
  macro_mon_mensual                       423 filas
  macro_riesgo                           5091 filas
  precios_empresas                     157455 filas
  ref_empresas                             36 filas
